In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
invoices = spark.read.table("medical_finance.bronze_layer.invoices")
purchase_orders = spark.read.table("medical_finance.bronze_layer.purchase_orders")
receipts = spark.read.table("medical_finance.bronze_layer.receipts")
invoice_items = spark.read.table("medical_finance.bronze_layer.invoice_items")

## 1. Clean  invoices table

In [0]:
invoices.display()

In [0]:
invoices.printSchema()

In [0]:
invoices = (invoices
    .withColumn("total_amount", regexp_replace(col("total_amount"), "[$,]", "").cast(DoubleType()))
    .withColumn("invoice_date", to_date(col("invoice_date"))) 
)

invoices.display()


## 2. Clean  purchase orders table

In [0]:
purchase_orders.limit(5).display()

In [0]:
purchase_orders= (purchase_orders
    .withColumn("po_date", to_date(col("po_date")))
    .withColumn("requested_ship_date", to_date(col("requested_ship_date")))
    .withColumn("subtotal", regexp_replace(col("subtotal"), "[$,]", "").cast(DoubleType()))
    .withColumn("shipping", regexp_replace(col("shipping"), "[$,]", "").cast(DoubleType()))
    .withColumn("total_amount", regexp_replace(col("total_amount"), "[$,]", "").cast(DoubleType()))
)

purchase_orders.limit(5).display()

## 3. Clean receipts table

In [0]:
receipts.display()

In [0]:
receipts.select("currency").distinct().display()

In [0]:
receipts =(receipts
    .withColumn("datetime", to_timestamp("datetime"))
    .withColumn("currency", regexp_replace(col("currency"), "[$]", "USD"))
    .withColumn("total_amount", regexp_replace(col("total_amount"), "[$,]", "").cast(DoubleType()))
)

receipts.display()

In [0]:
invoice_items.display()

In [0]:
invoice_items.printSchema()

## 4. Save Tables into silver layer

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_finance.silver_layer;

In [0]:
receipts.write.saveAsTable("medical_finance.silver_layer.receipts_cleaned")

In [0]:
invoices.write.saveAsTable("medical_finance.silver_layer.invoices_cleaned")
purchase_orders.write.saveAsTable("medical_finance.silver_layer.purchase_orders_cleaned")
invoice_items.write.saveAsTable("medical_finance.silver_layer.invoice_items_cleaned")